## 1. Prep Data for Combined UMAP

In [ ]:
import pandas as pd

In [ ]:
STRAT_XY_PATH = "../real_vase_data/profiles_stratified_120526.parquet"
ALL_REAL_XY_PATH = "../real_vase_data/all_profiles_120526.parquet"
GAME_XY_PATH = "../file_processing/processed/xy_data.parquet"
METADATA_PATH = "../real_vase_data/master_metadata.csv"
# AI_GAME_XY_PATH = "../ai_beauty/pca_llm_results/processed/ai_xy_data.parquet"

In [ ]:
all_real_xy = pd.read_parquet(ALL_REAL_XY_PATH)
strat_xy_df = pd.read_parquet(STRAT_XY_PATH)

In [ ]:
strat_gpp_nos = strat_xy_df["gpp_no"].unique()
real_xy_df = all_real_xy[all_real_xy["gpp_no"].isin(strat_gpp_nos)]
print(f"Real xy df shape: {real_xy_df.shape}")
print(f"Number of unique gpp_no in real xy df: {real_xy_df['gpp_no'].nunique()}")

In [ ]:

game_xy_df = pd.read_parquet(GAME_XY_PATH)

real_xy_df.rename(columns={'gpp_no': 'vase_id'}, inplace=True)

game_xy_df["vase_id"] = (
    game_xy_df["participant_id"].astype(str)
    + "::"
    + game_xy_df["generation_key"].astype(str)
    + "::"
    + game_xy_df["vase"].astype(str)
)

In [ ]:
real_xy_df['source'] = 'real'
real_xy_df.head()

In [ ]:
game_xy_df['source'] = 'game'
game_xy_df.head()

In [ ]:
# Filter so parent vases are excluded from the UMAP projection
game_xy_df = game_xy_df[game_xy_df['is_parent_vase'] != True]

In [ ]:
# ai_xy_df = pd.read_parquet(AI_GAME_XY_PATH)
# ai_xy_df['source'] = 'ai_game'
# ai_xy_df.head()

In [ ]:
# ai_xy_df["vase_id"] = (
#     ai_xy_df["participant_id"].astype(str)
#     + "::"
#     + ai_xy_df["generation_key"].astype(str)
#     + "::"
#     + ai_xy_df["vase"].astype(str)
# )
#
# ai_xy_df = ai_xy_df[ai_xy_df['is_parent_vase'] != True]

In [ ]:
REAL_PC_PATH = "../file_processing/pca_csv/pca_scores.csv"
GAME_PC_PATH = "../file_processing/processed/pc_data.parquet"
# AI_PC_PATH = "../ai_beauty/pca_llm_results/processed/ai_pc_data.parquet"

In [ ]:
real_df = pd.read_csv(REAL_PC_PATH)
real_df.rename(columns={'gpp_no': 'vase_id'}, inplace=True)
real_df['source'] = 'real'
real_df.drop(columns=['umap_x', 'umap_y'], inplace=True, errors='ignore')
print(f'Real latent DataFrame shape: {real_df.shape}')
real_df.head()

In [ ]:
game_df = pd.read_parquet(GAME_PC_PATH)

game_df['source'] = 'game'
game_df.drop(columns=['umap_x', 'umap_y'], inplace=True, errors='ignore')
print(f'Game latent DataFrame shape: {game_df.shape}')
game_df.head()

In [ ]:
# Filter out parent vases here too (mirrors the filter already applied to game_xy_df above),
# so the same vase carried forward across generations isn't counted or plotted multiple times.
game_df = game_df[game_df['is_parent_vase'] != True]

print(f'Game latent DataFrame shape: {game_df.shape}')
game_df.head()

In [ ]:
# ai_df = pd.read_parquet(AI_PC_PATH)
# ai_df['source'] = 'ai_game'
# ai_df.drop(columns=['umap_x', 'umap_y'], inplace=True, errors='ignore')
# print(f'AI latent DataFrame shape: {ai_df.shape}')
# ai_df.head()

### Pivot game_df from long to wide (one row per vase)

`pc_data.parquet` stores game vases in long format — one row per (vase, `pc_name`) pair, so each
vase has ~10 rows (PC1..PC10) all sharing the same metadata. `real_df` is already wide (one row per
vase). Concatenating them as-is into `master_df` silently multiplied every game vase's row count by
its number of PCs, which is what inflated the plotted game `N` far beyond the actual number of unique
vases. The cell below pivots `pc_name`/`pc_value` into `PC1..PCn` columns and collapses back to one
row per `vase_id` before `game_df` is used anywhere else.

In [ ]:
game_df["vase_id"] = (
    game_df["participant_id"].astype(str)
    + "::"
    + game_df["generation_key"].astype(str)
    + "::"
    + game_df["vase"].astype(str)
)

# ai_df["vase_id"] = (
#     ai_df["participant_id"].astype(str)
#     + "::"
#     + ai_df["generation_key"].astype(str)
#     + "::"
#     + ai_df["vase"].astype(str)
# )

# game_df arrives in long format: one row per (vase_id, pc_name) pair, with all other
# metadata columns repeated identically across a vase's PC rows. Pivot pc_name/pc_value
# into PC1..PCn columns and collapse back to one row per vase_id, so game_df has the same
# one-row-per-vase shape as real_df before it's used anywhere downstream (master_df, UMAP).
game_pc_pivot = (
    game_df
    .pivot_table(index='vase_id', columns='pc_name', values='pc_value', aggfunc='first')
    .reset_index()
)
game_pc_pivot.columns.name = None

game_meta_cols = [c for c in game_df.columns if c not in ('pc_name', 'pc_value')]
game_meta = game_df[game_meta_cols].drop_duplicates(subset='vase_id')

game_df = game_meta.merge(game_pc_pivot, on='vase_id', how='left')
print(f'Game DataFrame shape after pivot to one row per vase: {game_df.shape}')
game_df.head()

In [ ]:
# game_df is already one row per vase (pivoted above), so just pull out the PC + id columns.
pc_cols = sorted([c for c in game_df.columns if c.startswith('PC')])
game_pc_wide = game_df[pc_cols + ['vase_id']].copy()
game_pc_wide['source'] = 'game'
game_pc_wide['culture_general'] = None  # not available for game vases

# ai_pc_wide = ai_df[[c for c in ai_df.columns if c.startswith('PC')] + ['vase_id']].copy()
# ai_pc_wide['source'] = 'ai_game'
# ai_pc_wide['culture_general'] = None  # not available for ai game vases

game_pc_wide = game_pc_wide[pc_cols + ['vase_id', 'culture_general', 'source']]
# ai_pc_wide = ai_pc_wide[pc_cols + ['vase_id', 'culture_general', 'source']]

print(f'Game PC wide shape: {game_pc_wide.shape}')
# print(f'AI PC wide shape:   {ai_pc_wide.shape}')
print(f'Real PC shape:      {real_df.shape}')
game_pc_wide.head()

In [ ]:
real_latent_df = real_df[[ c for c in real_df.columns if c.startswith('PC') ]]
real_latent_df['source'] = 'real'
real_latent_df['vase_id'] = real_df['vase_id']
real_latent_df.head()

In [ ]:
game_latent_df = game_pc_wide[[ c for c in game_pc_wide.columns if c.startswith('PC') ]]
game_latent_df['source'] = 'game'
game_latent_df['vase_id'] = game_pc_wide['vase_id']
game_latent_df.head()

In [ ]:
# ai_latent_df = ai_pc_wide[[ c for c in ai_pc_wide.columns if c.startswith('PC') ]]
# ai_latent_df['source'] = 'ai_game'
# ai_latent_df['vase_id'] = ai_pc_wide['vase_id']
# ai_latent_df.head()

In [ ]:
latent_df = pd.concat([real_latent_df, game_latent_df], ignore_index=True) # Removed ai_latent_df
print(f'Combined latent DataFrame shape: {latent_df.shape}')
print(latent_df['source'].value_counts())
latent_df.head()

## 2. Optimise UMAP Projection

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import umap.umap_ as umap
from sklearn.manifold import trustworthiness
from tqdm import tqdm

In [ ]:
# [LATENT] Process all latent dimensions into an array for UMAP
latent_cols = [ c for c in latent_df.columns if c.startswith('PC') ]
latent_array = latent_df[latent_cols].values
print(f'Latent array shape: {latent_array.shape}')

### Normalize profiles before building the XY feature matrix

The neighbor-composition diagnostic below (section 2c) revealed that `real` and `game` vases were almost completely separated in the raw XY feature space. Tracing the two ingestion pipelines showed why: `real` profiles are pre-normalized to a fixed bounding-box scale (dominant axis span = 3.0, aspect preserved) before ever reaching this notebook, while `game` profiles are raw inverse-PCA reconstructions with no post-hoc rescaling — their per-vase coordinate ranges vary by roughly an order of magnitude. Since UMAP/k-NN here use Euclidean distance on raw coordinates, that scale mismatch alone was enough to explain most of the apparent separation.

`build_xy_feature_matrix()` below now applies the same per-vase centering + max-abs scaling to every profile, regardless of source, before flattening into the feature vector — putting `real` and `game` on a comparable scale for the first time.

In [ ]:
def normalize_profile(x, y):
    """Center on the profile's own mean and scale by max absolute coordinate,
    so every vase (regardless of source) lands on a comparable scale."""
    x = x - np.nanmean(x)
    y = y - np.nanmean(y)
    scale = max(np.nanmax(np.abs(x)), np.nanmax(np.abs(y)), 1e-9)
    return x / scale, y / scale

def build_xy_feature_matrix(df, id_col='vase_id', order_col='point_order', normalize=True):
    rows, ids, sources = [], [], []
    for (vase_id, source), grp in df.groupby([id_col, 'source'], sort=False):
        grp_sorted = grp.sort_values(order_col) if order_col in grp.columns else grp
        x = grp_sorted['x'].to_numpy(dtype=np.float32)
        y = grp_sorted['y'].to_numpy(dtype=np.float32)
        if normalize:
            x, y = normalize_profile(x, y)
        rows.append(np.concatenate([x, y]))
        ids.append(vase_id)
        sources.append(source)
    return np.stack(rows), pd.DataFrame({'vase_id': ids, 'source': sources})

xy_matrix_real, meta_real = build_xy_feature_matrix(real_xy_df)
xy_matrix_game, meta_game = build_xy_feature_matrix(game_xy_df)
# xy_matrix_ai, meta_ai = build_xy_feature_matrix(ai_xy_df)

# Toggle: use both sources or just game for testing
TEST_GAME_ONLY = False

if TEST_GAME_ONLY:
    xy_array = xy_matrix_game
    xy_meta_df = meta_game
else:
    xy_array = np.vstack([xy_matrix_real, xy_matrix_game])  # Removed xy_matrix_ai
    xy_meta_df = pd.concat([meta_real, meta_game], ignore_index=True)  # Removed meta_ai

print(f'XY feature matrix shape: {xy_array.shape}')
print(xy_meta_df['source'].value_counts())

In [ ]:
def umap_calibration(latent_matrix, sample_size=15000, n_neighbors_test = [15, 50, 150, 200], min_dist_test = [0.01, 0.1, 0.5]):
    np.random.seed(42)
    indices = np.random.choice(latent_matrix.shape[0], sample_size, replace=False)
    X_sample = latent_matrix[indices]
    
    fig, axes = plt.subplots(len(n_neighbors_test), len(min_dist_test), figsize=(18, 18))
    fig.subplots_adjust(hspace=0.3, wspace=0.3)
    
    best_score = 0
    best_params = {}
        
    for i, n_n in tqdm(enumerate(n_neighbors_test), total=len(n_neighbors_test)):
        for j, m_d in tqdm(enumerate(min_dist_test), total=len(min_dist_test)):
            print(f"   -> Testing: n_neighbors={n_n}, min_dist={m_d}")
            
            reducer = umap.UMAP(
                n_components=2, 
                n_neighbors=n_n, 
                min_dist=m_d, 
                metric='euclidean', 
                random_state=42
            )
            umap_coords = reducer.fit_transform(X_sample)
            
            # Trustworthiness measures to what extent the local structure is retained.
            # A score of 1.0 means the 2D map perfectly represents the true latent dimensions' local relationships.
            score = trustworthiness(X_sample, umap_coords, n_neighbors=n_n)
            
            if score > best_score:
                best_score = score
                best_params = {'n_neighbors': n_n, 'min_dist': m_d}
            
            ax = axes[i, j]
            ax.scatter(umap_coords[:, 0], umap_coords[:, 1], s=1, alpha=0.3, c='black')
            ax.set_title(f"n_neighbors: {n_n} | min_dist: {m_d}\nTrustworthiness: {score:.4f}", fontsize=12)
            ax.set_xticks([])
            ax.set_yticks([])
            
    plt.suptitle("UMAP Hyperparameter Topologies (15k Sample)", fontsize=20, y=0.92)
    plt.savefig("umap_calibration_grid.png", dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"   MATHEMATICAL OPTIMUM: n_neighbors={best_params['n_neighbors']}, min_dist={best_params['min_dist']} (Score: {best_score:.4f})")
    
    return best_params

In [ ]:
n_neighbors_test = [150, 200, 300] 
min_dist_test = [0.01, 0.02, 0.05, 0.1, 0.5]

In [ ]:
# optimal_params = umap_calibration(xy_array, sample_size=15000, n_neighbors_test=n_neighbors_test, min_dist_test=min_dist_test)
# print(f"Optimal UMAP Parameters: {optimal_params}")

In [ ]:
# n_n_opt = optimal_params['n_neighbors']
# m_d_opt = optimal_params['min_dist']

In [ ]:
reducer = umap.UMAP(
    n_components=2,
    n_neighbors=200,
    min_dist=0.02,
    metric='euclidean',
    random_state=42,
)
embedding = reducer.fit_transform(xy_array) 
print('UMAP embedding shape:', embedding.shape)

In [ ]:
umap_x = embedding[:, 0]
umap_y = embedding[:, 1]
xy_meta_df = xy_meta_df.copy()
xy_meta_df['umap_x'] = umap_x
xy_meta_df['umap_y'] = umap_y
xy_meta_df.head()

In [ ]:
master_df = pd.concat([real_df, game_df], ignore_index=True) # Removed ai_df
print(f'Master DataFrame shape: {master_df.shape}')
print(master_df.columns)
master_df.head()

In [ ]:
# Add the UMAP coordinates to the master DataFrame
# [XY] was: master_df.merge(latent_df[['vase_id', 'source', 'umap_x', 'umap_y']], ...)
master_df = master_df.merge(xy_meta_df[['vase_id', 'source', 'umap_x', 'umap_y']], on=['vase_id', 'source'], how='left')
print(f'Master DataFrame with UMAP shape: {master_df.shape}')
print(master_df.columns)
master_df.head()

## 2b. UMAP Projection on PC Latent Space (for comparison)

Fits a second UMAP using the same hyperparameters (`n_neighbors=200`, `min_dist=0.02`) but on the PCA latent vectors (`latent_array`) instead of the raw XY contour features, so the smoothness of the resulting landscape can be compared directly against the XY-based projection above.

In [ ]:
# [PC] Fit UMAP on the PCA latent space, using the same hyperparameters as the XY-based fit
# so the two projections are directly comparable.
reducer_pc = umap.UMAP(
    n_components=2,
    n_neighbors=200,
    min_dist=0.02,
    metric='euclidean',
    random_state=42,
)
embedding_pc = reducer_pc.fit_transform(latent_array)
print('PC UMAP embedding shape:', embedding_pc.shape)


In [ ]:
latent_meta_df = latent_df[['vase_id', 'source']].reset_index(drop=True)
latent_meta_df['umap_x_pc'] = embedding_pc[:, 0]
latent_meta_df['umap_y_pc'] = embedding_pc[:, 1]

master_df = master_df.merge(latent_meta_df, on=['vase_id', 'source'], how='left')
print(f'Master DataFrame with PC UMAP shape: {master_df.shape}')
master_df.head()


## 2c. Diagnostic: Neighbor Source Composition vs. Base Rate

`real` and `game` vases are imbalanced in the pooled dataset (~39% / ~61%). Check whether each vase's actual nearest neighbors reflect that base rate (sources are mixed / not separable) or deviate strongly from it (sources form distinct clusters). This uses the same feature space and `n_neighbors` that UMAP itself uses, via a plain k-NN search (not the UMAP embedding) so it's independent of any embedding distortion.

**Update:** running this on the raw (pre-normalization) XY features originally showed near-total separation (~97% / ~92% same-source neighbors), which turned out to be driven by a scale mismatch between the two ingestion pipelines rather than a genuine shape difference — see the normalization step added above in section 2. The XY diagnostic below now runs on the normalized `xy_array`, so it reflects whatever separation (if any) survives once both sources are on a comparable scale.

In [ ]:
from sklearn.neighbors import NearestNeighbors

def neighbor_source_composition(feature_matrix, meta_df, n_neighbors=200, sample_size=20000, random_state=42):
    """For a random sample of points, find their true k-NN (excluding self) in `feature_matrix`
    and report what fraction of those neighbors come from each source."""
    rng = np.random.default_rng(random_state)
    sample_idx = rng.choice(feature_matrix.shape[0], size=min(sample_size, feature_matrix.shape[0]), replace=False)

    nn = NearestNeighbors(n_neighbors=n_neighbors + 1, metric='euclidean')
    nn.fit(feature_matrix)
    _, indices = nn.kneighbors(feature_matrix[sample_idx])
    indices = indices[:, 1:]  # drop self

    sources = meta_df['source'].to_numpy()
    neighbor_sources = sources[indices]  # (sample_size, n_neighbors)

    out = meta_df.iloc[sample_idx].copy()
    out['frac_real_neighbors'] = (neighbor_sources == 'real').mean(axis=1)
    out['frac_game_neighbors'] = (neighbor_sources == 'game').mean(axis=1)
    return out

In [ ]:
# Base rate: the source mixture of the whole pool, i.e. what neighbor composition
# we'd expect by pure chance if source had no relationship to shape.
base_rate_real = (xy_meta_df['source'] == 'real').mean()
base_rate_game = (xy_meta_df['source'] == 'game').mean()
print(f"Base rate -> real: {base_rate_real:.1%}, game: {base_rate_game:.1%}")

In [ ]:
# Diagnostic on the XY contour feature space (same space + n_neighbors=200 as the XY UMAP fit)
xy_composition = neighbor_source_composition(xy_array, xy_meta_df, n_neighbors=200)

print("XY feature space — mean neighbor composition by vase's own source:")
print(xy_composition.groupby('source')[['frac_real_neighbors', 'frac_game_neighbors']].mean())
print(f"\n(Base rate for comparison -> real: {base_rate_real:.1%}, game: {base_rate_game:.1%})")

In [ ]:
# Same diagnostic on the PC latent feature space (same space + n_neighbors=200 as the PC UMAP fit)
base_rate_real_pc = (latent_meta_df['source'] == 'real').mean()
base_rate_game_pc = (latent_meta_df['source'] == 'game').mean()

pc_composition = neighbor_source_composition(latent_array, latent_meta_df, n_neighbors=200)

print("PC latent space — mean neighbor composition by vase's own source:")
print(pc_composition.groupby('source')[['frac_real_neighbors', 'frac_game_neighbors']].mean())
print(f"\n(Base rate for comparison -> real: {base_rate_real_pc:.1%}, game: {base_rate_game_pc:.1%})")

In [ ]:
# Visual summary: actual same-source neighbor fraction vs. base rate, for both feature spaces
fig, axes = plt.subplots(1, 2, figsize=(11, 5))

for ax, comp, (br_real, br_game), title in [
    (axes[0], xy_composition, (base_rate_real, base_rate_game), 'XY contour space'),
    (axes[1], pc_composition, (base_rate_real_pc, base_rate_game_pc), 'PC latent space'),
]:
    real_same = comp.loc[comp['source'] == 'real', 'frac_real_neighbors']
    game_same = comp.loc[comp['source'] == 'game', 'frac_game_neighbors']

    ax.boxplot([real_same, game_same], tick_labels=['real\n(own-source frac)', 'game\n(own-source frac)'])
    ax.axhline(br_real, color='#d1495b', linestyle='--', linewidth=1, label=f'real base rate ({br_real:.1%})')
    ax.axhline(br_game, color='#1f3b70', linestyle='--', linewidth=1, label=f'game base rate ({br_game:.1%})')
    ax.set_ylim(0, 1)
    ax.set_ylabel('fraction of k-NN from same source')
    ax.set_title(title)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 3. Plot UMAP Projections

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns
import numpy as np

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
colors = {'game': '#1f3b70', 'real': '#d1495b', 'ai_game': '#edae49'}
for source, sub in master_df.groupby('source'):
    ax.scatter(sub['umap_x'], sub['umap_y'], s=3, alpha=0.4, color=colors.get(source, 'gray'), label=source)
ax.legend(title='source')
ax.set_title(f'Universal UMAP of Vase space  (N = {len(master_df)})')
ax.set_xlabel('UMAP-1')
ax.set_ylabel('UMAP-2')
ax.set_aspect('equal')
plt.tight_layout()
plt.savefig("figures/umap_universal.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
colors  = {'game': '#1f3b70', 'real': '#d1495b', 'ai_game': '#edae49'}
sources = ['game', 'real']  # Removed ai_game
labels  = ['Game', 'Real']  # Removed AI Game

# Pre-compute plot limits from all data so every panel shares the same axis range
_valid = master_df.dropna(subset=['umap_x', 'umap_y'])
x_lim = (_valid['umap_x'].min(), _valid['umap_x'].max())
y_lim = (_valid['umap_y'].min(), _valid['umap_y'].max())

for ax, source, label in zip(axes, sources, labels):
    sub = master_df[master_df['source'] == source].dropna(subset=['umap_x', 'umap_y'])

    # All points in grey
    ax.scatter(
        _valid['umap_x'], _valid['umap_y'],
        s=1, alpha=0.12, color='#c0c0c0', rasterized=True,
    )
    # Highlighted source in colour
    ax.scatter(
        sub['umap_x'], sub['umap_y'],
        s=2, alpha=0.45, color=colors[source], rasterized=True,
    )

    ax.set_xlim(x_lim)
    ax.set_ylim(y_lim)
    ax.set_aspect('equal')
    ax.set_title(f'{label}  (N = {len(sub):,})', fontsize=13)
    ax.set_xlabel('UMAP-1')
    ax.set_ylabel('UMAP-2')
    ax.spines[['top', 'right']].set_visible(False)

fig.suptitle(f'UMAP by source  (total N = {len(_valid):,})', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig("figures/umap_by_source.png", dpi=300, bbox_inches='tight')
plt.show()

## 3b. Plot PC-based UMAP Projection (for comparison)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
colors = {'game': '#1f3b70', 'real': '#d1495b', 'ai_game': '#edae49'}
for source, sub in master_df.groupby('source'):
    ax.scatter(sub['umap_x_pc'], sub['umap_y_pc'], s=3, alpha=0.4, color=colors.get(source, 'gray'), label=source)
ax.legend(title='source')
ax.set_title(f'Universal UMAP of Vase space -- PC latent input  (N = {len(master_df)})')
ax.set_xlabel('UMAP-1')
ax.set_ylabel('UMAP-2')
ax.set_aspect('equal')
plt.tight_layout()
plt.savefig("figures/umap_universal_pc.png", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
colors  = {'game': '#1f3b70', 'real': '#d1495b', 'ai_game': '#edae49'}
sources = ['game', 'real']  # Removed ai_game
labels  = ['Game', 'Real']  # Removed AI Game

_valid = master_df.dropna(subset=['umap_x_pc', 'umap_y_pc'])
x_lim = (_valid['umap_x_pc'].min(), _valid['umap_x_pc'].max())
y_lim = (_valid['umap_y_pc'].min(), _valid['umap_y_pc'].max())

for ax, source, label in zip(axes, sources, labels):
    sub = master_df[master_df['source'] == source].dropna(subset=['umap_x_pc', 'umap_y_pc'])

    ax.scatter(
        _valid['umap_x_pc'], _valid['umap_y_pc'],
        s=1, alpha=0.12, color='#c0c0c0', rasterized=True,
    )
    ax.scatter(
        sub['umap_x_pc'], sub['umap_y_pc'],
        s=2, alpha=0.45, color=colors[source], rasterized=True,
    )

    ax.set_xlim(x_lim)
    ax.set_ylim(y_lim)
    ax.set_aspect('equal')
    ax.set_title(f'{label}  (N = {len(sub):,})', fontsize=13)
    ax.set_xlabel('UMAP-1')
    ax.set_ylabel('UMAP-2')
    ax.spines[['top', 'right']].set_visible(False)

fig.suptitle(f'PC-based UMAP by source  (total N = {len(_valid):,})', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig("figures/umap_by_source_pc.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Side-by-side comparison of the XY-based vs PC-based UMAP landscapes
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
colors = {'game': '#1f3b70', 'real': '#d1495b', 'ai_game': '#edae49'}

for source, sub in master_df.groupby('source'):
    axes[0].scatter(sub['umap_x'], sub['umap_y'], s=3, alpha=0.4, color=colors.get(source, 'gray'), label=source)
axes[0].set_title('UMAP fit on XY contour features')

for source, sub in master_df.groupby('source'):
    axes[1].scatter(sub['umap_x_pc'], sub['umap_y_pc'], s=3, alpha=0.4, color=colors.get(source, 'gray'), label=source)
axes[1].set_title('UMAP fit on PC latent features')

for ax in axes:
    ax.set_xlabel('UMAP-1')
    ax.set_ylabel('UMAP-2')
    ax.set_aspect('equal')
    ax.spines[['top', 'right']].set_visible(False)

axes[0].legend(title='source')
fig.suptitle('XY-based vs PC-based UMAP: landscape smoothness comparison', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("figures/umap_xy_vs_pc_comparison.png", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Build normalized contour dictionary directly from XY profile tables.
real_xy_for_plot = real_xy_df.rename(columns={'gpp_no': 'vase_id'})[['vase_id', 'x', 'y']].copy()
real_xy_for_plot['source'] = 'real'

game_cols = ['vase_id', 'x', 'y'] + (['coord_order'] if 'coord_order' in game_xy_df.columns else [])
game_xy_for_plot = game_xy_df[game_cols].copy()
game_xy_for_plot['source'] = 'game'

xy_for_plot = pd.concat([real_xy_for_plot, game_xy_for_plot], ignore_index=True)

xy_for_plot.head()

In [ ]:
contours_by_key = {}
for (src, vase_id), grp in xy_for_plot.groupby(['source', 'vase_id'], sort=False):
    if 'coord_order' in grp.columns:
        grp = grp.sort_values('coord_order')

    x = grp['x'].to_numpy(dtype=float)
    y = grp['y'].to_numpy(dtype=float)
    if len(x) < 3:
        continue

    # Ensure closed polygon for clean fills.
    if x[0] != x[-1] or y[0] != y[-1]:
        x = np.append(x, x[0])
        y = np.append(y, y[0])

    # Normalize each vase profile to fit the draw cell consistently.
    x = x - np.nanmean(x)
    y = y - np.nanmean(y)
    scale = max(np.nanmax(np.abs(x)), np.nanmax(np.abs(y)), 1e-9)
    x = x / scale
    y = y / scale

    contours_by_key[(src, vase_id)] = np.vstack([x, y])

In [ ]:
# ai_xy_df was not included when contours_by_key was built — add ai_game contours now
# for (src, vase_id), grp in ai_xy_df.groupby(['source', 'vase_id'], sort=False):
#     x = grp['x'].to_numpy(dtype=float)
#     y = grp['y'].to_numpy(dtype=float)
#     if len(x) < 3:
#         continue
#     if x[0] != x[-1] or y[0] != y[-1]:
#         x = np.append(x, x[0])
#         y = np.append(y, y[0])
#     x = x - np.nanmean(x)
#     y = y - np.nanmean(y)
#     scale = max(np.nanmax(np.abs(x)), np.nanmax(np.abs(y)), 1e-9)
#     x /= scale
#     y /= scale
#     contours_by_key[(src, vase_id)] = np.vstack([x, y])
#
# ai_keys = sum(1 for k in contours_by_key if k[0] == 'ai_game')
# print(f"contours_by_key total: {len(contours_by_key):,}  (ai_game: {ai_keys:,})")

In [ ]:
def plot_vase_umap_grid(plot_df, contours, title=None, save_path=None, N=50,
                        source_colors=None):
    if source_colors is None:
        source_colors = {'real': '#d1495b', 'game': '#1f3b70', 'ai_game': '#edae49'}

    df = plot_df.dropna(subset=['umap_x', 'umap_y']).reset_index(drop=True)
    xs = df['umap_x'].to_numpy()
    ys = df['umap_y'].to_numpy()

    x_min, x_max = xs.min(), xs.max()
    y_min, y_max = ys.min(), ys.max()
    cell_w = (x_max - x_min) / N
    cell_h = (y_max - y_min) / N

    col_idx = np.clip(((xs - x_min) / cell_w).astype(int), 0, N - 1)
    row_idx = np.clip(((ys - y_min) / cell_h).astype(int), 0, N - 1)

    cell_rep = {}
    for i, (ri, ci) in enumerate(zip(row_idx, col_idx)):
        cx = x_min + (ci + 0.5) * cell_w
        cy = y_min + (ri + 0.5) * cell_h
        d2 = (xs[i] - cx) ** 2 + (ys[i] - cy) ** 2
        if (ri, ci) not in cell_rep or d2 < cell_rep[(ri, ci)][0]:
            cell_rep[(ri, ci)] = (d2, i, cx, cy)

    draw_scale = cell_w * 0.30
    fig, ax = plt.subplots(figsize=(14, 14), dpi=150)
    ax.scatter(xs, ys, s=1, color='lightgrey', alpha=0.15, zorder=1, rasterized=True)

    for (ri, ci), (_, pt_idx, cx, cy) in cell_rep.items():
        row = df.iloc[pt_idx]
        src, vase_id = row['source'], row['vase_id']
        contour = contours.get((src, vase_id))
        if contour is None:
            continue
        ax.fill(cx + contour[0] * draw_scale, cy + contour[1] * draw_scale,
                color=source_colors.get(src, 'gray'), alpha=0.85, edgecolor='none', zorder=2)

    legend_handles = [Patch(facecolor=c, edgecolor='none', label=s)
                      for s, c in source_colors.items() if s in df['source'].values]
    ax.legend(handles=legend_handles, title='source', frameon=False)
    ax.set_aspect('equal')
    ax.set_xlabel('UMAP 1', fontweight='bold', labelpad=10)
    ax.set_ylabel('UMAP 2', fontweight='bold', labelpad=10)
    ax.spines[['top', 'right']].set_visible(False)
    if title:
        ax.set_title(title)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
plot_vase_umap_grid(master_df, contours_by_key, save_path='vase_umap_grid.svg')

In [ ]:
plot_vase_umap_grid(master_df[master_df['source'] == 'game'], contours_by_key, save_path='game_vase_umap_grid.svg')

In [ ]:
plot_vase_umap_grid(master_df[master_df['source'] == 'real'], contours_by_key, save_path='real_vase_umap_grid.svg')

In [ ]:
# plot_vase_umap_grid(master_df[master_df['source'] == 'ai_game'], contours_by_key, save_path='ai_game_vase_umap_grid.svg')

## 4. Save the UMAP Dataframes

In [ ]:
import os

In [ ]:
SAVE_DIR = "umap_projections"
REAL_SAVE_PATH = os.path.join(SAVE_DIR, "real_vase_umap_projection.parquet")
GAME_SAVE_PATH = os.path.join(SAVE_DIR, "game_vase_umap_projection.parquet")
MASTER_SAVE_PATH = os.path.join(SAVE_DIR, "combined_vase_umap_projection.parquet")

In [ ]:
os.makedirs(SAVE_DIR, exist_ok=True)
master_df.to_parquet(MASTER_SAVE_PATH, index=False)

In [ ]:
real_umap_df = master_df[master_df['source'] == 'real']
real_umap_df.drop([col for col in real_umap_df.columns if col.startswith('PC')], axis=1, inplace=True, errors='ignore')
real_umap_df.drop(columns=['source', "generation", "vase", "gen_selected_vase", "elapsed_time", 
                           "ancestral_region", "youth_region", "disability", "user_id", "play_counter", 
                           "round", "user_round_generation", "potter", "age", "gender", "sexuality"], inplace=True, errors='ignore')
real_umap_df.head()

In [ ]:
game_umap_df = master_df[master_df['source'] == 'game']
game_umap_df.drop([col for col in game_umap_df.columns if col.startswith('PC')], axis=1, inplace=True, errors='ignore')
game_umap_df.drop(columns=["culture_general", "culture_specific", "continent_made_found", "country_made_found"], inplace=True, errors='ignore')
game_umap_df.head()

In [ ]:
# ai_umap_df = master_df[master_df['source'] == 'ai_game']
# ai_umap_df.drop([col for col in ai_umap_df.columns if col.startswith('PC')], axis=1, inplace=True, errors='ignore')
# ai_umap_df.drop(columns=["culture_general", "culture_specific", "continent_made_found", "country_made_found"], inplace=True, errors='ignore')
# ai_umap_df.head()

In [ ]:
real_umap_df.to_parquet(REAL_SAVE_PATH, index=False)
game_umap_df.to_parquet(GAME_SAVE_PATH, index=False)
# ai_umap_df.to_parquet(os.path.join(SAVE_DIR, "ai_game_vase_umap_projection.parquet"), index=False)

In [ ]:
# Separate lightweight files holding just the two UMAP projections (all sources combined),
# for reuse without pulling in the full metadata/PC columns.
XY_UMAP_SAVE_PATH = os.path.join(SAVE_DIR, "xy_vase_umap_projection.parquet")
PC_UMAP_SAVE_PATH = os.path.join(SAVE_DIR, "pc_vase_umap_projection.parquet")


In [ ]:
xy_meta_df[['vase_id', 'source', 'umap_x', 'umap_y']].to_parquet(XY_UMAP_SAVE_PATH, index=False)
latent_meta_df[['vase_id', 'source', 'umap_x_pc', 'umap_y_pc']].to_parquet(PC_UMAP_SAVE_PATH, index=False)


### Full versions (all metadata + PC scores) for downstream analysis

Same two projections as above, but keeping every column in `master_df` (demographics, `generation_key`, `phase`, `is_selected_vase`, PC scores, etc.) instead of just `vase_id`/`source`/coords. Each file keeps only the coordinate columns for its own projection so there's no ambiguity about which UMAP a downstream script is using.

In [ ]:
XY_UMAP_FULL_SAVE_PATH = os.path.join(SAVE_DIR, "xy_vase_umap_projection_full.parquet")
PC_UMAP_FULL_SAVE_PATH = os.path.join(SAVE_DIR, "pc_vase_umap_projection_full.parquet")

xy_umap_full_df = master_df.drop(columns=['umap_x_pc', 'umap_y_pc'], errors='ignore')
pc_umap_full_df = master_df.drop(columns=['umap_x', 'umap_y'], errors='ignore')

xy_umap_full_df.to_parquet(XY_UMAP_FULL_SAVE_PATH, index=False)
pc_umap_full_df.to_parquet(PC_UMAP_FULL_SAVE_PATH, index=False)

print(f'XY full UMAP saved: {xy_umap_full_df.shape} -> {XY_UMAP_FULL_SAVE_PATH}')
print(f'PC full UMAP saved: {pc_umap_full_df.shape} -> {PC_UMAP_FULL_SAVE_PATH}')
